# The kernel trick
**Machine Learning · Week 4 (Kernels and support vector machines) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/ml/w4d3.html

The kernel trick: an SVM only needs inner products between points, so a **kernel** $k(x, x')$ can stand in
for a feature map we never compute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC

def phi(v):
    """The feature map hidden inside the degree-2 polynomial kernel."""
    x1, x2 = v
    return np.array([x1 ** 2, np.sqrt(2) * x1 * x2, x2 ** 2])

x, z = np.array([1.0, 2.0]), np.array([3.0, -0.5])
print("phi(x) . phi(z) =", round(phi(x) @ phi(z), 6), "  <- 3 features each, then a dot product")
print("(x . z)^2       =", round((x @ z) ** 2, 6), "  <- one dot product in 2D, then a square")

Same number, much less work. For degree $d$ in $p$ dimensions, the hidden feature space has a number of
features that explodes, while the kernel always costs one dot product and one power.

## The RBF kernel, three ways
$k(x, x') = e^{-\gamma\|x - x'\|^2}$: its hidden feature space has infinitely many dimensions.

In [ ]:
from sklearn.datasets import make_circles
from matplotlib.colors import LinearSegmentedColormap

apricot_blue = LinearSegmentedColormap.from_list("apricot_blue", ["#ff8a50", "#ffffff", "#5577f5"])
Xc, yc = make_circles(n_samples=200, noise=0.08, factor=0.45, random_state=1)
gx, gy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.7), sharey=True, layout="constrained")
for ax, gamma in zip(axes, [0.1, 2, 100]):
    m = SVC(kernel="rbf", gamma=gamma, C=1.0).fit(Xc, yc)
    f = m.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, f, levels=20, cmap=apricot_blue, alpha=0.5)
    ax.contour(gx, gy, f, levels=[0], colors="#2c2842", linewidths=1.8)
    ax.scatter(Xc[:, 0], Xc[:, 1], c=np.where(yc == 0, "#ff8a50", "#5577f5"), s=10)
    ax.set_title(f"gamma = {gamma}\naccuracy {m.score(Xc, yc):.2f}, {len(m.support_)} support vectors")
    ax.set_aspect("equal")
plt.show()

## A prediction is a weighted vote of similarities
$f(x) = \sum_{i \in \text{SV}} \alpha_i y_i\, k(x_i, x) + b$. scikit-learn stores $\alpha_i y_i$ in `dual_coef_`.
Let's rebuild its decision for one new point by hand.

In [ ]:
m = SVC(kernel="rbf", gamma=2, C=1.0).fit(Xc, yc)
new = np.array([0.2, 0.3])
k = np.exp(-2 * np.sum((m.support_vectors_ - new) ** 2, axis=1))     # similarity to every support vector
by_hand = m.dual_coef_[0] @ k + m.intercept_[0]
print("by hand:     ", round(by_hand, 6))
print("scikit-learn:", round(m.decision_function([new])[0], 6))
print("prediction:  ", "blue (class 1)" if by_hand > 0 else "orange (class 0)")

## Your turn: which kernel fits the circles?
A linear SVM fails on the circles. Try `SVC(kernel="poly", degree=2)`. Why is degree 2 enough?
(Think about the features hidden in it, from cell 1.)

In [ ]:
lin = SVC(kernel="linear").fit(Xc, yc)
print("linear kernel:  ", lin.score(Xc, yc))
poly_acc = None    # TODO: fit SVC(kernel="poly", degree=2, coef0=1) and score it on (Xc, yc)
print("degree-2 kernel:", poly_acc)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
poly = SVC(kernel="poly", degree=2, coef0=1).fit(Xc, yc)
print("linear kernel:  ", round(lin.score(Xc, yc), 3))
print("degree-2 kernel:", round(poly.score(Xc, yc), 3))
print("Its hidden features include x1^2 and x2^2, and a circle is x1^2 + x2^2 = r^2:")
print("a straight cut in that hidden space is a circle in ours (Monday's lift, done implicitly).")